NOTICE: Code is optimized for the A100 GPU

# 1. Installing, importing libraries and making configurations

In [1]:
!pip install uv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 111.8 MB/s eta 0:00:00


In [2]:
!uv pip install torch numpy matplotlib lightning huggingface_hub triton pandas optuna optuna-integration[pytorch_lightning]

Using Python 3.12.13 environment at: /usr
Resolved 79 packages in 523ms
Prepared 7 packages in 98ms
Installed 7 packages in 13ms
 + colorlog==6.10.1
 + lightning==2.6.1
 + lightning-utilities==0.15.3
 + optuna==4.8.0
 + optuna-integration==4.8.0
 + pytorch-lightning==2.6.1
 + torchmetrics==1.9.0


In [3]:
!uv pip install mamba-ssm --no-build-isolation-package mamba-ssm

Using Python 3.12.13 environment at: /usr
Resolved 56 packages in 194ms
Prepared 1 package in 8ms
Installed 1 package in 2ms
Prepared 1 package without build isolation in 12.82s
Installed 1 package in 1ms
 + mamba-ssm==2.3.1
 + ninja==1.13.0


In [4]:
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint


import lightning as L
from datasets import load_dataset
from huggingface_hub import HfApi, hf_hub_download, login
from lightning.pytorch.loggers import CSVLogger
from torch.utils.data import DataLoader, Dataset, TensorDataset, random_split

from mamba_ssm import Mamba

import optuna
from optuna.integration import PyTorchLightningPruningCallback

from mamba_ssm.ops.selective_scan_interface import selective_scan_fn

In [5]:
NUM_WORKERS = 4

NUM_EPOCHS = 5

# 2. Models developed

In [6]:
def label_smoothed_bce_loss(logits, labels, eps=0.1):
    ce_hard = F.cross_entropy(logits, labels)
    log_probs = F.log_softmax(logits, dim=-1)
    ce_soft = -log_probs.mean(dim=-1).mean()
    return (1.0 - eps) * ce_hard + eps * ce_soft

In [7]:
def get_optimizer(model, lr=1e-3, base_wd=0.01, delta_wd=1e-3):
    delta_params = []
    other_params = []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if 'delta_fixed' in name:
            delta_params.append(param)
        else:
            other_params.append(param)
    param_groups = [
        {'params': other_params, 'weight_decay': base_wd,  'name': 'default'},
        {'params': delta_params, 'weight_decay': delta_wd, 'name': 'delta_fixed'},
    ]
    return torch.optim.AdamW(param_groups, lr=lr)

In [8]:
class RMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-8):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))

    def forward(self, x):
        rms = x.pow(2).mean(-1, keepdim=True).add(self.eps).sqrt()
        return x / rms * self.weight

In [9]:
class ResidualBlock(nn.Module):
    def __init__(self, d_model, N, expand=2, **kwargs):
        super().__init__()
        self.norm = RMSNorm(d_model)
        self.ssm  = FixedSSMBlock(d_model=d_model, d_state=N, expand=expand)

    def forward(self, x):
        return x + self.ssm(self.norm(x))

In [10]:
class FixedSSMBlock(nn.Module):
    def __init__(self, d_model, d_state=16, d_conv=4, expand=2):
        super().__init__()
        self.d_inner = d_model * expand
        self.d_state = d_state

        self.in_proj  = nn.Linear(d_model, 2 * self.d_inner, bias=False)
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)

        self.conv1d = nn.Conv1d(
            self.d_inner, self.d_inner,
            kernel_size=d_conv, padding=d_conv - 1, groups=self.d_inner,
        )

        A = torch.arange(1, d_state + 1, dtype=torch.float32)
        A = A.unsqueeze(0).expand(self.d_inner, -1)
        self.A_log = nn.Parameter(torch.log(A))
        self.D = nn.Parameter(torch.ones(self.d_inner))

        # FIXED (non-input-dependent) B, C, delta
        self.B_fixed = nn.Parameter(torch.randn(d_state) * 0.02)
        self.C_fixed = nn.Parameter(torch.randn(d_state) * 0.02)
        self.delta_fixed = nn.Parameter(torch.zeros(self.d_inner))

    def forward(self, x):
        batch, length, _ = x.shape

        xz = self.in_proj(x).transpose(1, 2)
        x_inner, z = xz.chunk(2, dim=1)

        x_inner = self.conv1d(x_inner)[..., :length]
        x_inner = F.silu(x_inner)

        A = -torch.exp(self.A_log)
        delta = (F.softplus(self.delta_fixed)
                 .unsqueeze(0).unsqueeze(-1)
                 .expand(batch, -1, length).contiguous())
        B = (self.B_fixed
             .unsqueeze(0).unsqueeze(-1)
             .expand(batch, -1, length).contiguous())
        C = (self.C_fixed
             .unsqueeze(0).unsqueeze(-1)
             .expand(batch, -1, length).contiguous())

        delta, B, C = delta.to(x_inner.dtype), B.to(x_inner.dtype), C.to(x_inner.dtype)

        y = selective_scan_fn(
            x_inner, delta, A, B, C,
            D=self.D, z=z, delta_softplus=False,
        )
        return self.out_proj(y.transpose(1, 2))

In [11]:
class MambaClassifier(L.LightningModule):
    def __init__(
        self,
        vocab_size: int,
        d_model: int = 128,
        n_layers: int = 2,
        N: int = 16,
        num_classes: int = 2,
        lr: float = 1e-3,
        label_smoothing: float = 0.0,
        delta_wd: float = 0.0,
        expand: int = 2,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.save_hyperparameters()
        self.lr = lr
        self.label_smoothing = label_smoothing
        self.delta_wd = delta_wd
        self.num_classes = num_classes

        self.embedding = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([
            ResidualBlock(d_model, N, expand=expand) for _ in range(n_layers)
        ])
        self.norm_f = RMSNorm(d_model)

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, num_classes),
        )

        # Compile Mamba layers for fused Triton kernels (replaces causal-conv1d)
        for i, layer in enumerate(self.layers):
            self.layers[i] = torch.compile(layer)

    def forward(self, input_ids, attention_mask=None):
        x = self.embedding(input_ids)               # (B, L, D)

        for layer in self.layers:
            x = layer(x)
        x = self.norm_f(x)

        if attention_mask is not None:
            mask = attention_mask.unsqueeze(-1).float()
            x = (x * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        else:
            x = x.mean(dim=1)

        return self.classifier(x)

    def _shared_step(self, batch, stage):
        logits = self(batch['input_ids'], batch['attention_mask'])
        labels = batch['labels']

        if self.label_smoothing > 0:
            loss = label_smoothed_bce_loss(logits, labels, eps=self.label_smoothing)
        else:
            loss = F.cross_entropy(logits, labels)

        preds = logits.argmax(dim=-1)
        acc = (preds == labels).float().mean()

        self.log(f'{stage}_loss', loss, prog_bar=True)
        self.log(f'{stage}_acc',  acc,  prog_bar=True)
        return loss

    def training_step(self, batch, batch_idx):
        return self._shared_step(batch, 'train')

    def validation_step(self, batch, batch_idx):
        return self._shared_step(batch, 'val')

    def test_step(self, batch, batch_idx):
        return self._shared_step(batch, 'test')

    def configure_optimizers(self):
        if self.delta_wd > 0:
            return get_optimizer(
                self, lr=self.lr, base_wd=0.01, delta_wd=self.delta_wd
            )
        return torch.optim.AdamW(self.parameters(), lr=self.lr, weight_decay=0.01)

# 3. Data and running each model in its benchmark

## 3.0 Data

In [12]:
repo_id = "monteirot/lra-benchmarks"

files = ['listops.pt', 'imdb_lra.pt', 'acl_retrieval_lra.pt', 'cifar10_sequential_lra.pt']

for f in files:
    print(f"Downloading {f}...")
    hf_hub_download(repo_id=repo_id, filename=f, repo_type="dataset", local_dir=".")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


listops.pt:   0%|          | 0.00/175M [00:00<?, ?B/s]

imdb_lra.pt:   0%|          | 0.00/132M [00:00<?, ?B/s]

acl_retrieval_lra.pt:   0%|          | 0.00/274M [00:00<?, ?B/s]

cifar10_sequential_lra.pt:   0%|          | 0.00/124M [00:00<?, ?B/s]

In [13]:
def make_trainer():
    return L.Trainer(
        max_epochs=6,
        accelerator='gpu' if torch.cuda.is_available() else 'cpu',
        devices=1,
        precision='bf16-mixed',
        logger=CSVLogger(save_dir="logs", name="lra-benchmarks"),
    )

In [14]:
def make_hpo_trainer(max_epochs=9):
    """Lightweight trainer for HPO — no checkpointing, no logging."""
    return L.Trainer(
        max_epochs=max_epochs,
        accelerator='gpu' if torch.cuda.is_available() else 'cpu',
        devices=1,
        precision='bf16-mixed',
        enable_checkpointing=False,
        logger=False,
    )

## 3.1 ListOpsDataset

In [15]:
class ListOpsDataset(Dataset):
    def __init__(self, data, max_len=2048):
        self.data = data
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        tokens, label = self.data[idx]

        # Pad or truncate
        if len(tokens) < self.max_len:
            mask = [1] * len(tokens) + [0] * (self.max_len - len(tokens))
            tokens = tokens + [0] * (self.max_len - len(tokens))
        else:
            tokens = tokens[:self.max_len]
            mask = [1] * self.max_len

        return {
            'input_ids': torch.tensor(tokens, dtype=torch.long),
            'attention_mask': torch.tensor(mask, dtype=torch.long),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [16]:
data_listops = torch.load('listops.pt', weights_only=False)

### 3.1.1 Finding the bets hyperparameters

In [17]:
listops_search_space = {
    "d_model":    [64, 128],
    "n_layers":   [2, 4],
    "N":          [8, 16],
    "lr":         [1e-3, 5e-4],
    "batch_size": [4, 8],
}

def objective(trial):
    d_model    = trial.suggest_categorical("d_model",    listops_search_space["d_model"])
    n_layers   = trial.suggest_categorical("n_layers",   listops_search_space["n_layers"])
    N          = trial.suggest_categorical("N",          listops_search_space["N"])
    lr         = trial.suggest_categorical("lr",         listops_search_space["lr"])
    batch_size = trial.suggest_categorical("batch_size", listops_search_space["batch_size"])

    model = MambaClassifier(
        vocab_size=data_listops['vocab_size'],
        d_model=d_model,
        n_layers=n_layers,
        N=N,
        num_classes=num_classes,
        lr=lr,
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size,
                            num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)

    trainer = make_hpo_trainer(max_epochs=NUM_EPOCHS)
    try:
        trainer.fit(model, train_loader, val_loader)
        return trainer.callback_metrics["val_acc"].item()
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        raise optuna.TrialPruned()

In [18]:
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.GridSampler(listops_search_space),
    study_name="listops-mamba-grid",
)

[I 2026-04-10 22:06:15,863] A new study created in memory with name: listops-mamba-grid


In [19]:
# Datasets & loaders for Optuna study
train_dataset = ListOpsDataset(data_listops['train'].data)
val_dataset = ListOpsDataset(data_listops['val'].data)

# Calculate num_classes from the training data
train_labels = [item[1] for item in data_listops['train'].data]
num_classes = max(train_labels) + 1

In [20]:
study.optimize(objective, n_trials=2)

INFO: Using bfloat16 Automatic Mixed Precision (AMP)
INFO:lightning.pytorch.utilities.rank_zero:Using bfloat16 Automatic Mixed Precision (AMP)
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: You are using a CUDA device ('NVIDIA A100-SXM4-40GB') that has Tensor Cores. To prop

┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ embedding  │ Embedding  │  1.1 K │ train │     0 │
│ 1 │ layers     │ ModuleList │  106 K │ train │     0 │
│ 2 │ norm_f     │ RMSNorm    │     64 │ train │     0 │
│ 3 │ classifier │ Sequential │  4.8 K │ train │     0 │
└───┴────────────┴────────────┴────────┴───────┴───────┘

Trainable params: 112 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 112 K                                                                                                
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 37                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

...

Output()

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

INFO: `Trainer.fit` stopped: `max_epochs=5` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=5` reached.


[I 2026-04-10 23:12:27,661] Trial 1 finished with value: 0.42149999737739563 and parameters: {'d_model': 128, 'n_layers': 2, 'N': 16, 'lr': 0.0005, 'batch_size': 4}. Best is trial 1 with value: 0.42149999737739563.


### 3.1.2 Training the model with the best hyperpaparemters

In [21]:
best = study.best_trial.params

print(best)

{'d_model': 128, 'n_layers': 2, 'N': 16, 'lr': 0.0005, 'batch_size': 4}


In [22]:
model_to_find_num_parameters = MambaClassifier(
    vocab_size=data_listops['vocab_size'],
    d_model=best["d_model"],
    n_layers=best["n_layers"],
    N=best["N"],
    num_classes=num_classes,
    lr=best["lr"],
)

total_params = sum(p.numel() for p in model_to_find_num_parameters.parameters() if p.requires_grad)

print("Best trial:", study.best_trial.params)
print("Best val_acc:", study.best_trial.value)

print(f"Best val_acc: {study.best_trial.value:.4f} | Params: {study.best_trial.params}")

print(f"Model Parameters: {total_params}")

Best trial: {'d_model': 128, 'n_layers': 2, 'N': 16, 'lr': 0.0005, 'batch_size': 4}
Best val_acc: 0.42149999737739563
Best val_acc: 0.4215 | Params: {'d_model': 128, 'n_layers': 2, 'N': 16, 'lr': 0.0005, 'batch_size': 4}
Model Parameters: 228810


In [23]:
# Datasets & loaders
train_dataset = ListOpsDataset(data_listops['train'].data)
val_dataset = ListOpsDataset(data_listops['val'].data)

train_loader = DataLoader(train_dataset, batch_size=best["batch_size"], shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
val_loader = DataLoader(val_dataset, batch_size=best["batch_size"], num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)

# Calculate num_classes from the training data
train_labels = [item[1] for item in data_listops['train'].data]
num_classes = max(train_labels) + 1

# Model
model_listops = MambaClassifier(
    vocab_size=data_listops['vocab_size'],
    d_model=best['d_model'],
    n_layers=best['n_layers'],
    N=best['N'],
    num_classes=num_classes,
    lr=best['lr'],
)

trainer = make_trainer()

trainer.fit(model_listops, train_loader, val_loader)

test_dataset = ListOpsDataset(data_listops['test'].data)
test_loader = DataLoader(test_dataset, batch_size=32, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)

trainer.test(model_listops, test_loader)

INFO: Using bfloat16 Automatic Mixed Precision (AMP)
INFO:lightning.pytorch.utilities.rank_zero:Using bfloat16 Automatic Mixed Precision (AMP)
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https:/

┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ embedding  │ Embedding  │  2.2 K │ train │     0 │
│ 1 │ layers     │ ModuleList │  208 K │ train │     0 │
│ 2 │ norm_f     │ RMSNorm    │    128 │ train │     0 │
│ 3 │ classifier │ Sequential │ 17.8 K │ train │     0 │
└───┴────────────┴────────────┴────────┴───────┴───────┘

Trainable params: 228 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 228 K                                                                                                
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 23                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

INFO: `Trainer.fit` stopped: `max_epochs=6` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=6` reached.


INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.4065000116825104     │
│         test_loss         │    1.5130336284637451     │
└───────────────────────────┴───────────────────────────┘

[{'test_loss': 1.5130336284637451, 'test_acc': 0.4065000116825104}]

## 3.2 IMDbDataset

In [24]:
class IMDbDataset(Dataset):
    """PyTorch Dataset for byte-level IMDb classification."""
    def __init__(self, data, max_len=4096):
        self.data = data
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        tokens, label = self.data[idx]

        # Truncate or pad to max_len
        if len(tokens) < self.max_len:
            mask = [1] * len(tokens) + [0] * (self.max_len - len(tokens))
            tokens = tokens + [0] * (self.max_len - len(tokens))
        else:
            tokens = tokens[:self.max_len]
            mask = [1] * self.max_len

        return {
            'input_ids': torch.tensor(tokens, dtype=torch.long),
            'attention_mask': torch.tensor(mask, dtype=torch.long),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [25]:
data_imdb = torch.load('imdb_lra.pt', weights_only=False)

### 3.2.1 Finding the bets hyperparameters

In [26]:
imdb_search_space = {
    "d_model":    [64, 128],
    "n_layers":   [2, 4],
    "N":          [8, 16],
    "lr":         [1e-3, 5e-4],
    "batch_size": [4, 8],
}


def objective_imdb(trial):
    d_model    = trial.suggest_categorical("d_model",    imdb_search_space["d_model"])
    n_layers   = trial.suggest_categorical("n_layers",   imdb_search_space["n_layers"])
    N          = trial.suggest_categorical("N",          imdb_search_space["N"])
    lr         = trial.suggest_categorical("lr",         imdb_search_space["lr"])
    batch_size = trial.suggest_categorical("batch_size", imdb_search_space["batch_size"])

    model = MambaClassifier(
        vocab_size=data_imdb['vocab_size'],
        d_model=d_model,
        n_layers=n_layers,
        N=N,
        num_classes=num_classes,
        lr=lr,
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size,
                            num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)

    trainer = make_hpo_trainer(max_epochs=NUM_EPOCHS)
    trainer.fit(model, train_loader, val_loader)
    return trainer.callback_metrics["val_acc"].item()

In [27]:
study_imdb = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.GridSampler(imdb_search_space),
    study_name="imdb-mamba-grid",
)

[I 2026-04-11 00:00:58,405] A new study created in memory with name: imdb-mamba-grid


In [28]:
# Datasets & loaders for Optuna study
train_dataset = IMDbDataset(data_imdb['train'].data)
val_dataset = IMDbDataset(data_imdb['val'].data)

# Calculate num_classes from the training data
train_labels = [item[1] for item in data_imdb['train'].data]
num_classes = max(train_labels) + 1

In [29]:
study_imdb.optimize(objective_imdb, n_trials=2)

INFO: Using bfloat16 Automatic Mixed Precision (AMP)
INFO:lightning.pytorch.utilities.rank_zero:Using bfloat16 Automatic Mixed Precision (AMP)
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:L

┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ embedding  │ Embedding  │ 16.4 K │ train │     0 │
│ 1 │ layers     │ ModuleList │  106 K │ train │     0 │
│ 2 │ norm_f     │ RMSNorm    │     64 │ train │     0 │
│ 3 │ classifier │ Sequential │  4.3 K │ train │     0 │
└───┴────────────┴────────────┴────────┴───────┴───────┘

Trainable params: 127 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 127 K                                                                                                
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 37                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

...

Output()

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

W0411 00:07:47.112000 544 torch/_dynamo/convert_frame.py:1676] [0/8] torch._dynamo hit config.recompile_limit (8)
W0411 00:07:47.112000 544 torch/_dynamo/convert_frame.py:1676] [0/8]    function: 'forward' (/tmp/ipykernel_544/1493110823.py:7)
W0411 00:07:47.112000 544 torch/_dynamo/convert_frame.py:1676] [0/8]    last reason: 0/7: GLOBAL_STATE changed: grad_mode 
W0411 00:07:47.112000 544 torch/_dynamo/convert_frame.py:1676] [0/8] To log all recompilation reasons, use TORCH_LOGS="recompiles".
W0411 00:07:47.112000 544 torch/_dynamo/convert_frame.py:1676] [0/8] To diagnose recompilation issues, see https://pytorch.org/docs/main/compile/programming_model.recompilation.html
INFO: `Trainer.fit` stopped: `max_epochs=5` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=5` reached.


[I 2026-04-11 00:16:48,363] Trial 1 finished with value: 0.86080002784729 and parameters: {'d_model': 128, 'n_layers': 2, 'N': 16, 'lr': 0.0005, 'batch_size': 4}. Best is trial 0 with value: 0.864799976348877.


In [30]:
best = study_imdb.best_trial.params

print(best)

model_to_find_num_parameters = MambaClassifier(
        vocab_size=data_imdb['vocab_size'],
        d_model=best["d_model"],
        n_layers=best["n_layers"],
        N=best["N"],
        num_classes=num_classes,
        lr=best["lr"],
    )


# 2. Calculate total trainable parameters
total_params = sum(p.numel() for p in model_to_find_num_parameters.parameters() if p.requires_grad)

print("Best trial:", study_imdb.best_trial.params)
print("Best val_acc:", study_imdb.best_trial.value)

print(f"Best val_acc: {study_imdb.best_trial.value:.4f} | Params: {study_imdb.best_trial.params}")

print(f"Model Parameters: {total_params}")

{'d_model': 64, 'n_layers': 4, 'N': 8, 'lr': 0.0005, 'batch_size': 8}
Best trial: {'d_model': 64, 'n_layers': 4, 'N': 8, 'lr': 0.0005, 'batch_size': 8}
Best val_acc: 0.864799976348877
Best val_acc: 0.8648 | Params: {'d_model': 64, 'n_layers': 4, 'N': 8, 'lr': 0.0005, 'batch_size': 8}
Model Parameters: 127106


### 3.2.2 Running the best model with the best hyperparameters

In [31]:
# Datasets & loaders
train_dataset = IMDbDataset(data_imdb['train'].data)
val_dataset = IMDbDataset(data_imdb['val'].data)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
val_loader = DataLoader(val_dataset, batch_size=32, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)

# Calculate num_classes from the training data
train_labels = [item[1] for item in data_imdb['train'].data]
num_classes = max(train_labels) + 1

# Model
model_imdb = MambaClassifier(
    vocab_size=data_imdb['vocab_size'],
    d_model=best['d_model'],
    n_layers=best['n_layers'],
    N=best['N'],
    num_classes=num_classes,
    lr=best['lr'],
)

In [32]:
trainer = make_trainer()

INFO: Using bfloat16 Automatic Mixed Precision (AMP)
INFO:lightning.pytorch.utilities.rank_zero:Using bfloat16 Automatic Mixed Precision (AMP)
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [33]:
trainer.fit(model_imdb, train_loader, val_loader)

INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/model_summary/model_summary.py:242: Precision bf16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ embedding  │ Embedding  │ 16.4 K │ train │     0 │
│ 1 │ layers     │ ModuleList │  106 K │ train │     0 │
│ 2 │ norm_f     │ RMSNorm    │     64 │ train │     0 │
│ 3 │ classifier │ Sequential │  4.3 K │ train │     0 │
└───┴────────────┴────────────┴────────┴───────┴───────┘

Trainable params: 127 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 127 K                                                                                                
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 37                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

INFO: `Trainer.fit` stopped: `max_epochs=6` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=6` reached.


In [34]:
test_dataset = IMDbDataset(data_imdb['test'].data)
test_loader = DataLoader(test_dataset, batch_size=32, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)

In [35]:
trainer.test(model_imdb, test_loader)

INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.8530399799346924     │
│         test_loss         │    0.36087292432785034    │
└───────────────────────────┴───────────────────────────┘

[{'test_loss': 0.36087292432785034, 'test_acc': 0.8530399799346924}]

## 3.3 ACLRetricalDataset

In [36]:
class ACLRetrievalDataset(Dataset):
    """PyTorch Dataset for ACL document retrieval."""
    def __init__(self, data, max_len=8001):  # 4000 + 1 (sep) + 4000
        self.data = data
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        tokens, label = self.data[idx]

        # Truncate or pad
        if len(tokens) < self.max_len:
            mask = [1] * len(tokens) + [0] * (self.max_len - len(tokens))
            tokens = tokens + [0] * (self.max_len - len(tokens))
        else:
            tokens = tokens[:self.max_len]
            mask = [1] * self.max_len

        return {
            'input_ids': torch.tensor(tokens, dtype=torch.long),
            'attention_mask': torch.tensor(mask, dtype=torch.long),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [37]:
data_retrieval = torch.load('acl_retrieval_lra.pt', weights_only=False)

### 3.3.1 Finding the bets hyperparameters

In [38]:
retrieval_search_space = {
    "d_model":    [64, 128],
    "n_layers":   [2, 4],
    "N":          [8, 16],
    "lr":         [1e-3, 5e-4],
    "batch_size": [4, 8],
}

def objective_retrival(trial):
    d_model    = trial.suggest_categorical("d_model",    retrieval_search_space["d_model"])
    n_layers   = trial.suggest_categorical("n_layers",   retrieval_search_space["n_layers"])
    N          = trial.suggest_categorical("N",          retrieval_search_space["N"])
    lr         = trial.suggest_categorical("lr",         retrieval_search_space["lr"])
    batch_size = trial.suggest_categorical("batch_size", retrieval_search_space["batch_size"])

    model = MambaClassifier(
        vocab_size=data_retrieval['vocab_size'],
        d_model=d_model,
        n_layers=n_layers,
        N=N,
        num_classes=num_classes,
        lr=lr,
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size,
                            num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)

    trainer = make_hpo_trainer(max_epochs=NUM_EPOCHS)
    trainer.fit(model, train_loader, val_loader)
    return trainer.callback_metrics["val_acc"].item()

In [39]:
study_retrival = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.GridSampler(retrieval_search_space),
    study_name="retrieval-mamba-grid",
)

[I 2026-04-11 00:19:23,403] A new study created in memory with name: retrieval-mamba-grid


In [40]:
# Datasets & loaders for Optuna study
train_dataset = ACLRetrievalDataset(data_retrieval['train'].data)
val_dataset = ACLRetrievalDataset(data_retrieval['val'].data)

# Calculate num_classes from the training data
train_labels = [item[1] for item in data_retrieval['train'].data]
num_classes = max(train_labels) + 1

In [41]:
study_retrival.optimize(objective_retrival, n_trials=2)

INFO: Using bfloat16 Automatic Mixed Precision (AMP)
INFO:lightning.pytorch.utilities.rank_zero:Using bfloat16 Automatic Mixed Precision (AMP)
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:L

┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ embedding  │ Embedding  │ 16.4 K │ train │     0 │
│ 1 │ layers     │ ModuleList │  106 K │ train │     0 │
│ 2 │ norm_f     │ RMSNorm    │     64 │ train │     0 │
│ 3 │ classifier │ Sequential │  4.3 K │ train │     0 │
└───┴────────────┴────────────┴────────┴───────┴───────┘

Trainable params: 127 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 127 K                                                                                                
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 37                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

INFO: `Trainer.fit` stopped: `max_epochs=5` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=5` reached.


...

┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ embedding  │ Embedding  │ 32.9 K │ train │     0 │
│ 1 │ layers     │ ModuleList │  208 K │ train │     0 │
│ 2 │ norm_f     │ RMSNorm    │    128 │ train │     0 │
│ 3 │ classifier │ Sequential │ 16.8 K │ train │     0 │
└───┴────────────┴────────────┴────────┴───────┴───────┘

Trainable params: 258 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 258 K                                                                                                
Total estimated model params size (MB): 1                                                                          
Modules in train mode: 23                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

INFO: `Trainer.fit` stopped: `max_epochs=5` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=5` reached.


[I 2026-04-11 00:33:38,168] Trial 1 finished with value: 0.4964999854564667 and parameters: {'d_model': 128, 'n_layers': 2, 'N': 16, 'lr': 0.0005, 'batch_size': 4}. Best is trial 0 with value: 0.5.


In [42]:
best = study_retrival.best_trial.params
#chnage this
model_to_find_num_parameters = MambaClassifier(
        vocab_size=data_retrieval['vocab_size'],
        d_model=best["d_model"],
        n_layers=best["n_layers"],
        N=best["N"],
        num_classes=num_classes,
        lr=best["lr"],
    )

# 2. Calculate total trainable parameters
total_params = sum(p.numel() for p in model_to_find_num_parameters.parameters() if p.requires_grad)

print("Best trial:", study_retrival.best_trial.params)
print("Best val_acc:", study_retrival.best_trial.value)

print(f"Best val_acc: {study_retrival.best_trial.value:.4f} | Params: {study_retrival.best_trial.params}")

print(f"Model Parameters: {total_params}")

Best trial: {'d_model': 64, 'n_layers': 4, 'N': 8, 'lr': 0.0005, 'batch_size': 8}
Best val_acc: 0.5
Best val_acc: 0.5000 | Params: {'d_model': 64, 'n_layers': 4, 'N': 8, 'lr': 0.0005, 'batch_size': 8}
Model Parameters: 127106


### 3.3.2 Training model with best hyperparameters

In [43]:
# Datasets & loaders
train_dataset = ACLRetrievalDataset(data_retrieval['train'].data)
val_dataset = ACLRetrievalDataset(data_retrieval['val'].data)

train_loader = DataLoader(train_dataset, batch_size=best["batch_size"], shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
val_loader = DataLoader(val_dataset, batch_size=best["batch_size"], num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)

# Calculate num_classes from the training data
train_labels = [item[1] for item in data_retrieval['train'].data]
num_classes = max(train_labels) + 1

# Model
model_acl = MambaClassifier(
    vocab_size=data_retrieval['vocab_size'],
    d_model=best['d_model'],
    n_layers=best['n_layers'],
    N=best['N'],
    num_classes=num_classes,
    lr=best['lr'],
)

trainer = make_trainer()

trainer.fit(model_acl, train_loader, val_loader)

test_dataset = ACLRetrievalDataset(data_retrieval['test'].data)
test_loader = DataLoader(test_dataset, batch_size=32, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)

trainer.test(model_acl, test_loader)

INFO: Using bfloat16 Automatic Mixed Precision (AMP)
INFO:lightning.pytorch.utilities.rank_zero:Using bfloat16 Automatic Mixed Precision (AMP)
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https:/

┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ embedding  │ Embedding  │ 16.4 K │ train │     0 │
│ 1 │ layers     │ ModuleList │  106 K │ train │     0 │
│ 2 │ norm_f     │ RMSNorm    │     64 │ train │     0 │
│ 3 │ classifier │ Sequential │  4.3 K │ train │     0 │
└───┴────────────┴────────────┴────────┴───────┴───────┘

Trainable params: 127 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 127 K                                                                                                
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 37                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

INFO: `Trainer.fit` stopped: `max_epochs=6` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=6` reached.


INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.4984999895095825     │
│         test_loss         │    0.6950390338897705     │
└───────────────────────────┴───────────────────────────┘

[{'test_loss': 0.6950390338897705, 'test_acc': 0.4984999895095825}]

## 3.4 CIFAR10SequentialDataset

In [44]:
class CIFAR10SequentialDataset(Dataset):
    """PyTorch Dataset for sequential CIFAR-10 classification."""
    def __init__(self, data, seq_len=1024):
        self.data = data
        self.seq_len = seq_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        # Handle both dict and tuple formats
        if isinstance(item, dict):
            tokens = item['input_ids']
            label = item['labels']
        else:
            tokens = item[0]
            label = item[-1]

        # Convert tensors to lists if needed (for padding logic)
        if isinstance(tokens, torch.Tensor):
            tokens = tokens.tolist()
        if isinstance(label, torch.Tensor):
            label = label.item()

        # Pad or truncate to seq_len
        if len(tokens) < self.seq_len:
            mask = [1] * len(tokens) + [0] * (self.seq_len - len(tokens))
            tokens = tokens + [0] * (self.seq_len - len(tokens))
        else:
            tokens = tokens[:self.seq_len]
            mask = [1] * self.seq_len

        return {
            'input_ids': torch.tensor(tokens, dtype=torch.long),
            'attention_mask': torch.tensor(mask, dtype=torch.long),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [45]:
data_cifar10_sequential = torch.load('cifar10_sequential_lra.pt', weights_only=False)

### 3.4.1 Finding the bets hyperparameters

In [46]:
cifar_search_space = {
    "d_model":    [64, 128],
    "n_layers":   [2, 4],
    "N":          [8, 16],
    "lr":         [1e-3, 5e-4],
    "batch_size": [8, 16],
}

def objective_cifar(trial):
    d_model    = trial.suggest_categorical("d_model",    cifar_search_space["d_model"])
    n_layers   = trial.suggest_categorical("n_layers",   cifar_search_space["n_layers"])
    N          = trial.suggest_categorical("N",          cifar_search_space["N"])
    lr         = trial.suggest_categorical("lr",         cifar_search_space["lr"])
    batch_size = trial.suggest_categorical("batch_size", cifar_search_space["batch_size"])

    model = MambaClassifier(
        vocab_size=data_cifar10_sequential['vocab_size'],
        d_model=d_model,
        n_layers=n_layers,
        N=N,
        num_classes=num_classes,
        lr=lr,
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size,
                            num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)

    trainer = make_hpo_trainer(max_epochs=NUM_EPOCHS)
    trainer.fit(model, train_loader, val_loader)
    return trainer.callback_metrics["val_acc"].item()

In [47]:
study_cifar = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.GridSampler(cifar_search_space),
    study_name="cifar-mamba-grid",
)

[I 2026-04-11 00:41:06,482] A new study created in memory with name: cifar-mamba-grid


In [48]:
# Datasets & loaders for Optuna study
train_dataset = CIFAR10SequentialDataset(data_cifar10_sequential['train'].data)
val_dataset = CIFAR10SequentialDataset(data_cifar10_sequential['val'].data)

# Calculate num_classes from the training data
train_labels = [item[1] for item in data_cifar10_sequential['train'].data]
num_classes = max(train_labels) + 1

In [49]:
study_cifar.optimize(objective_cifar, n_trials=2)

INFO: Using bfloat16 Automatic Mixed Precision (AMP)
INFO:lightning.pytorch.utilities.rank_zero:Using bfloat16 Automatic Mixed Precision (AMP)
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:L

┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ embedding  │ Embedding  │ 16.4 K │ train │     0 │
│ 1 │ layers     │ ModuleList │  106 K │ train │     0 │
│ 2 │ norm_f     │ RMSNorm    │     64 │ train │     0 │
│ 3 │ classifier │ Sequential │  4.8 K │ train │     0 │
└───┴────────────┴────────────┴────────┴───────┴───────┘

Trainable params: 127 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 127 K                                                                                                
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 37                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

INFO: `Trainer.fit` stopped: `max_epochs=5` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=5` reached.


...

┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ embedding  │ Embedding  │ 32.9 K │ train │     0 │
│ 1 │ layers     │ ModuleList │  208 K │ train │     0 │
│ 2 │ norm_f     │ RMSNorm    │    128 │ train │     0 │
│ 3 │ classifier │ Sequential │ 17.8 K │ train │     0 │
└───┴────────────┴────────────┴────────┴───────┴───────┘

Trainable params: 259 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 259 K                                                                                                
Total estimated model params size (MB): 1                                                                          
Modules in train mode: 23                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

INFO: `Trainer.fit` stopped: `max_epochs=5` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=5` reached.


[I 2026-04-11 00:57:24,153] Trial 1 finished with value: 0.3709999918937683 and parameters: {'d_model': 128, 'n_layers': 2, 'N': 16, 'lr': 0.0005, 'batch_size': 8}. Best is trial 0 with value: 0.3725999891757965.


In [50]:
best = study_cifar.best_trial.params

#chnage this
model_to_find_num_parameters = MambaClassifier(
        vocab_size=data_cifar10_sequential['vocab_size'],
        d_model=best["d_model"],
        n_layers=best["n_layers"],
        N=best["N"],
        num_classes=num_classes,
        lr=best["lr"],
    )


# 2. Calculate total trainable parameters
total_params = sum(p.numel() for p in model_to_find_num_parameters.parameters() if p.requires_grad)

print("Best trial:", study_cifar.best_trial.params)
print("Best val_acc:", study_cifar.best_trial.value)

print(f"Best val_acc: {study_cifar.best_trial.value:.4f} | Params: {study_cifar.best_trial.params}")

print(f"Model Parameters: {total_params}")

Best trial: {'d_model': 64, 'n_layers': 4, 'N': 8, 'lr': 0.0005, 'batch_size': 16}
Best val_acc: 0.3725999891757965
Best val_acc: 0.3726 | Params: {'d_model': 64, 'n_layers': 4, 'N': 8, 'lr': 0.0005, 'batch_size': 16}
Model Parameters: 127626


### 3.4.2 Training model with best hyperparameters

In [51]:
# Datasets & loaders
train_dataset = CIFAR10SequentialDataset(data_cifar10_sequential['train'].data)
val_dataset = CIFAR10SequentialDataset(data_cifar10_sequential['val'].data)

train_loader = DataLoader(train_dataset, batch_size=best["batch_size"], shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
val_loader = DataLoader(val_dataset, batch_size=best["batch_size"], num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)

# Calculate num_classes from the training data
train_labels = [item[1] for item in data_cifar10_sequential['train'].data]
num_classes = max(train_labels) + 1

# Model
model_cifar = MambaClassifier(
    vocab_size=data_cifar10_sequential['vocab_size'],
    d_model=best['d_model'],
    n_layers=best['n_layers'],
    N=best['N'],
    num_classes=num_classes,
    lr=best['lr'],
)

trainer = make_trainer()

trainer.fit(model_cifar, train_loader, val_loader)

test_dataset = CIFAR10SequentialDataset(data_cifar10_sequential['test'].data)
test_loader = DataLoader(test_dataset, batch_size=32, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)

trainer.test(model_cifar, test_loader)

INFO: Using bfloat16 Automatic Mixed Precision (AMP)
INFO:lightning.pytorch.utilities.rank_zero:Using bfloat16 Automatic Mixed Precision (AMP)
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https:/

┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ embedding  │ Embedding  │ 16.4 K │ train │     0 │
│ 1 │ layers     │ ModuleList │  106 K │ train │     0 │
│ 2 │ norm_f     │ RMSNorm    │     64 │ train │     0 │
│ 3 │ classifier │ Sequential │  4.8 K │ train │     0 │
└───┴────────────┴────────────┴────────┴───────┴───────┘

Trainable params: 127 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 127 K                                                                                                
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 37                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

INFO: `Trainer.fit` stopped: `max_epochs=6` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=6` reached.


INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.3783000111579895     │
│         test_loss         │    1.7214796543121338     │
└───────────────────────────┴───────────────────────────┘

[{'test_loss': 1.7214796543121338, 'test_acc': 0.3783000111579895}]

# 4. Seeing results

In [52]:
print("\n" + "=" * 70)
print("  GRID SEARCH RESULTS SUMMARY (Optuna GridSampler)")
print("=" * 70)

for name, study_obj in [
    ("ListOps",     study),
    ("IMDb",        study_imdb),
    ("Retrieval",   study_retrival),
    ("CIFAR-10",    study_cifar),
]:
    print(f"\n  {name:15s}  best val_acc = {study_obj.best_trial.value:.4f}")
    print(f"  {'':15s}  params: {study_obj.best_trial.params}")

print("\n" + "=" * 70)


  GRID SEARCH RESULTS SUMMARY (Optuna GridSampler)

  ListOps          best val_acc = 0.4215
                   params: {'d_model': 128, 'n_layers': 2, 'N': 16, 'lr': 0.0005, 'batch_size': 4}

  IMDb             best val_acc = 0.8648
                   params: {'d_model': 64, 'n_layers': 4, 'N': 8, 'lr': 0.0005, 'batch_size': 8}

  Retrieval        best val_acc = 0.5000
                   params: {'d_model': 64, 'n_layers': 4, 'N': 8, 'lr': 0.0005, 'batch_size': 8}

  CIFAR-10         best val_acc = 0.3726
                   params: {'d_model': 64, 'n_layers': 4, 'N': 8, 'lr': 0.0005, 'batch_size': 16}

